# 🧾 Verificación reproducible en Kaggle — Agente de Órdenes de Compra SAP

    Este notebook no es un backend aparte : clona el repositorio y ejecuta, sobre Kaggle, el mismo backend TypeScript (`src/`, herramientas `oc_*`, SAP simulado) para verificarlo de punta a punta:
    
    1. Instala Bun y las dependencias,
    2. Corre las 13 pruebas y el `demo.ts` (los 6 casos, sin modelo de lenguaje)
    3. *(Opcional)* Prueba el agente contra OpenAI usando un Secret de Kaggle,
    4. Muestra los artefactos generados (`control.csv`, payloads, evidencia)
    Requiere **Internet On** en la configuración del notebook. Para el paso 3, agrega el Secret `OPENAI_API_KEY` en *Add-ons → Secrets*."

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/Dockerfile
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/demo.ts
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/.gitignore
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/README.md
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/SOLUCION.md
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/.dockerignore
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/.env.example
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/tsconfig.json
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/package.json
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/modulo/agent.md
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/modulo/tools/oc.ts
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/modulo/skill/ordenes-compra/SKILL.md
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/scripts/empaquetar-modulo.ts
/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/scripts/modulo-lib.ts
/kaggle/input/datasets/sebasg

In [2]:
%%bash
ls -R /kaggle/input | head -40

/kaggle/input:
datasets

/kaggle/input/datasets:
sebasgp1411

/kaggle/input/datasets/sebasgp1411:
reto-03v2

/kaggle/input/datasets/sebasgp1411/reto-03v2:
reto-03

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03:
agent
demo.ts
Dockerfile
fixtures
modulo
package.json
README.md
scripts
SOLUCION.md
src
test
tsconfig.json
web

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/agent:
prompt.md

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/fixtures:
reto-03

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/fixtures/reto-03:
maestros
solicitudes

/kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03/fixtures/reto-03/maestros:
centros-costo.json
condiciones-pago.json


In [3]:
%%bash
set -e
rm -rf /kaggle/working/reto-03
cp -r /kaggle/input/datasets/sebasgp1411/reto-03v2/reto-03 /kaggle/working/reto-03
ls /kaggle/working/reto-03

agent
demo.ts
Dockerfile
fixtures
modulo
package.json
README.md
scripts
SOLUCION.md
src
test
tsconfig.json
web


In [4]:
%%bash
curl -fsSL https://bun.sh/install | bash
$HOME/.bun/bin/bun --version

bun was installed successfully to ~/.bun/bin/bun 

Added "~/.bun/bin" to $PATH in "~/.bashrc" 

To get started, run: 

  source /root/.bashrc 
  bun --help 
1.4.3


######################################################################## 100.0%


# Pruebas y demo determinista (sin modelo)

In [5]:
   %%bash
   export PATH=$HOME/.bun/bin:$PATH
   cd /kaggle/working/reto-03
   bun run test && bun run demo.ts

Pruebas del agente
  ✔ 1. esConfirmacion distingue confirmar de negar
  ✔ 2. CA3: el modelo no puede confirmar por su cuenta; solo el siguiente mensaje del usuario
  ✔ 3. Si el usuario dice que no, la OC no se crea aunque el modelo insista
  ✔ 4. Un 'confirmo' sin pregunta previa no crea nada
  ✔ 5. Bloqueos: el caso sol-002 nunca se crea
  ✔ 6. CA5: error del proveedor LLM se muestra claro y la sesión sigue viva
  ✔ 7. CA1: tope de iteraciones por turno
  ✔ 8. Costo: tope de tokens por sesión
  ✔ 9. Costo: tope global de tokens
  ✔ 10. API: health, chat, sesión, validaciones y clave de acceso
  ✔ 11. Adaptador OpenAI: formato de petición/respuesta, errores sin filtrar la clave y timeout
  ✔ 12. Módulo (bonus): modulo/ coincide con las fuentes de la app

12 pruebas OK

━━━━━━━━ sol-001 ━━━━━━━━
  apta: true | retroactiva: false
  bloqueos:       —
  confirmaciones: —
  ✅ OC 4500000001 | evidencia: out/sol-001/aprobacion.txt

━━━━━━━━ sol-002 ━━━━━━━━
  apta: false | retroactiva: false


$ bun run test/agente.test.ts


# 3) (Opcional) Prueba real contra el modelo
Consume tokens de tu clave de OpenAI. Ejecuta el prompt de demo del reto sobre sol-004, luego confirma, y luego procesa sol-002 (bloqueada).

In [6]:
   from kaggle_secrets import UserSecretsClient
   import os, subprocess
   env = {**os.environ, "OPENAI_API_KEY": UserSecretsClient().get_secret("OPENAI_API_KEY"),
          "PATH": os.path.expanduser("~/.bun/bin") + ":" + os.environ["PATH"]}
   r = subprocess.run(["bun", "run", "probar:modelo"], cwd="/kaggle/working/reto-03",
                      env=env, capture_output=True, text=True)
   print(r.stdout, r.stderr[-1500:])

Modelo: openai/gpt-5-mini

👤 Procesa la solicitud "sol-004". Muéstrame la OC como quedaría en SAP, qué validaciones pasó y cuáles no, y no la crees hasta que yo lo confirme.
   🔧 oc_leer_paquete({"caso":"sol-004"}) -> Paquete leído (SOL-2026-004); faltantes: []
   🔧 oc_validar({"caso":"sol-004"}) -> apta=true · bloqueos: — · confirmaciones: RC5 · retroactiva=false
   🔧 oc_construir_payload({"caso":"sol-004"}) -> Payload construido y validado. Trazabilidad: out/sol-004/trazabilidad.json

🤖 Adjunto cómo quedaría la OC en SAP (sin crear) y el estado de las validaciones. No creo la OC hasta que confirmes.

Resumen de la OC
- Proveedor (código SAP): Cloud Andina S.A.S. (100310) — NIT: 901222333
- Moneda: COP
- Condiciones de pago: Z000
- Aprobador: dgarcia@periferia-ficticia.com (fecha: 2026-08-26T18:45:00-05:00)
- Posiciones:
  - Pos. 10 — Descripción: Bolsa de 100 horas de arquitectura de  
    Cantidad: 100 — Precio unitario: 250000 — Centro de costo: CC-1010 — Subárea: Desarrollo — IVA: